# NYC Yellow Taxi Trip Analysis (Jan 2024)

Analyzing ~1 GB of NYC taxi trip data to test Cash caching with real-world data:
- Fare distributions & tip percentages by time-of-day
- Popular pickup/dropoff zones
- Trip duration patterns & heatmaps
- Merge with taxi zone lookup table

In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import time
import os
import urllib.request

print(f"pandas {pd.__version__}, numpy {np.__version__}")
print(f"CWD: {os.getcwd()}")

In [ ]:
# Download NYC Yellow Taxi data (Jan 2024) - ~40 MB Parquet file
# Also download the taxi zone lookup table
data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)

taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
taxi_file = os.path.join(data_dir, 'yellow_tripdata_2024-01.parquet')

zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zones_file = os.path.join(data_dir, 'taxi_zone_lookup.csv')

for url, filepath in [(taxi_url, taxi_file), (zones_url, zones_file)]:
    if not os.path.exists(filepath):
        print(f"Downloading {os.path.basename(filepath)}...")
        t0 = time.time()
        urllib.request.urlretrieve(url, filepath)
        print(f"  Done in {time.time()-t0:.1f}s, size: {os.path.getsize(filepath)/1024/1024:.1f} MB")
    else:
        print(f"Already have {os.path.basename(filepath)} ({os.path.getsize(filepath)/1024/1024:.1f} MB)")

In [ ]:
# Load the raw data
t0 = time.time()
df = pd.read_parquet(taxi_file)
elapsed = time.time() - t0
print(f"Loaded {len(df):,} rows in {elapsed:.2f}s")
print(f"Memory: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['tpep_pickup_datetime'].min()} to {df['tpep_pickup_datetime'].max()}")

In [ ]:
# Data cleaning & feature engineering  
t0 = time.time()

# Remove outliers and invalid data
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['total_amount'] > 0]

# BUG FIX: Filter to only January 2024 data (noticed dirty dates from 2002, 2009)
df = df[(df['tpep_pickup_datetime'] >= '2024-01-01') & (df['tpep_pickup_datetime'] < '2024-02-01')]

# Extract time features
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day'] = df['tpep_pickup_datetime'].dt.day_name()
df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
df['trip_duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

# Remove unreasonable durations
df = df[(df['trip_duration_min'] > 0.5) & (df['trip_duration_min'] < 180)]

# Tip percentage (only for credit card payments - payment_type == 1)
df['tip_pct'] = np.where(df['payment_type'] == 1, df['tip_amount'] / df['fare_amount'] * 100, np.nan)

elapsed = time.time() - t0
print(f"Cleaned & engineered in {elapsed:.2f}s")
print(f"Remaining rows: {len(df):,} ({len(df)/2964624*100:.1f}% of original)")
print(f"Memory: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")

In [ ]:
# Fare and tip analysis by hour of day
t0 = time.time()

hourly_stats = df.groupby('pickup_hour').agg(
    avg_fare=('fare_amount', 'mean'),
    avg_tip_pct=('tip_pct', 'mean'),
    avg_distance=('trip_distance', 'mean'),
    avg_duration=('trip_duration_min', 'mean'),
    trip_count=('fare_amount', 'count'),
    total_revenue=('total_amount', 'sum')
).round(2)

elapsed = time.time() - t0
print(f"Hourly aggregation computed in {elapsed:.2f}s")
print(f"\nHourly Stats:")
print(hourly_stats.to_string())

In [ ]:
# Load taxi zone lookup and merge
zones = pd.read_csv(zones_file)
print(f"Zones: {len(zones)} rows")
print(zones.head())

# Merge pickup zones
df = df.merge(zones[['LocationID', 'Borough', 'Zone']], 
              left_on='PULocationID', right_on='LocationID', how='left')
df = df.rename(columns={'Borough': 'PU_Borough', 'Zone': 'PU_Zone'})
df = df.drop(columns=['LocationID'])

# Merge dropoff zones  
df = df.merge(zones[['LocationID', 'Borough', 'Zone']], 
              left_on='DOLocationID', right_on='LocationID', how='left')
df = df.rename(columns={'Borough': 'DO_Borough', 'Zone': 'DO_Zone'})
df = df.drop(columns=['LocationID'])

print(f"\nMerged DataFrame: {len(df):,} rows x {len(df.columns)} columns")
print(f"Memory: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")

In [ ]:
# Top 15 most popular routes
t0 = time.time()

# Deliberate bug: using wrong column name (will fix later to test cache invalidation)
top_routes = df.groupby(['PU_Zone', 'DO_Zone']).agg(
    trips=('fare_amount', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip=('tip_pct', 'mean'),
    avg_duration=('trip_duration_min', 'mean')
).sort_values('trips', ascending=False).head(15).round(2)

elapsed = time.time() - t0
print(f"Top routes computed in {elapsed:.2f}s")
print(f"\nTop 15 Most Popular Routes:")
print(top_routes.to_string())

In [ ]:
# Borough-level trip analysis
t0 = time.time()

borough_stats = df.groupby('PU_Borough').agg(
    trips=('fare_amount', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip_pct=('tip_pct', 'mean'),
    avg_distance=('trip_distance', 'mean'),
    total_revenue=('total_amount', 'sum')
).sort_values('trips', ascending=False).round(2)

# Cross-borough flow matrix
flow_matrix = df.groupby(['PU_Borough', 'DO_Borough']).size().unstack(fill_value=0)

elapsed = time.time() - t0
print(f"Borough analysis computed in {elapsed:.2f}s")
print(f"\nBorough Stats:")
print(borough_stats.to_string())
print(f"\nCross-Borough Flow Matrix:")
print(flow_matrix.to_string())

In [ ]:
# Trip duration analysis by day of week
t0 = time.time()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_stats = df.groupby('pickup_day').agg(
    avg_duration=('trip_duration_min', 'mean'),
    median_duration=('trip_duration_min', 'median'),
    avg_fare=('fare_amount', 'mean'),
    trip_count=('fare_amount', 'count')
).reindex(day_order).round(2)

# Revenue per day
daily_revenue = df.groupby([df['tpep_pickup_datetime'].dt.date]).agg(
    total_revenue=('total_amount', 'sum'),
    trip_count=('fare_amount', 'count')
).round(2)

elapsed = time.time() - t0
print(f"Day analysis computed in {elapsed:.2f}s")
print(f"\nDay-of-Week Stats:")
print(day_stats.to_string())
print(f"\nDaily Revenue (first 10 days):")
print(daily_revenue.head(10).to_string())

In [ ]:
# Fare distribution analysis - compute quantiles and stats for the full 2.86M rows
t0 = time.time()

fare_stats = {
    'mean': df['fare_amount'].mean(),
    'median': df['fare_amount'].median(),
    'std': df['fare_amount'].std(),
    'p5': df['fare_amount'].quantile(0.05),
    'p25': df['fare_amount'].quantile(0.25),
    'p75': df['fare_amount'].quantile(0.75),
    'p95': df['fare_amount'].quantile(0.95),
    'p99': df['fare_amount'].quantile(0.99),
}

# Fare histogram data (100 bins)
fare_hist, fare_edges = np.histogram(df['fare_amount'].dropna(), bins=100, range=(0, 100))

# Tip analysis for credit card only
cc_mask = df['payment_type'] == 1
tip_stats = {
    'cc_trips': cc_mask.sum(),
    'mean_tip_pct': df.loc[cc_mask, 'tip_pct'].mean(),
    'median_tip_pct': df.loc[cc_mask, 'tip_pct'].median(),
    'zero_tip_pct': (df.loc[cc_mask, 'tip_amount'] == 0).mean() * 100,
}

elapsed = time.time() - t0
print(f"Fare distribution computed in {elapsed:.2f}s")
print(f"\nFare Statistics:")
for k, v in fare_stats.items():
    print(f"  {k}: ${v:.2f}")
print(f"\nTip Statistics (credit card only):")
for k, v in tip_stats.items():
    print(f"  {k}: {v:.2f}")
print(f"\nHistogram peak: ${fare_edges[np.argmax(fare_hist)]:.0f}-${fare_edges[np.argmax(fare_hist)+1]:.0f} ({fare_hist.max():,} trips)")

In [ ]:
# Airport trip analysis (JFK = LocationID 132, LaGuardia = 138, EWR = 1)
t0 = time.time()

airport_ids = {'JFK': 132, 'LaGuardia': 138, 'EWR': 1}

airport_results = {}
for name, loc_id in airport_ids.items():
    to_airport = df[df['DOLocationID'] == loc_id]
    from_airport = df[df['PULocationID'] == loc_id]
    
    airport_results[name] = {
        'trips_to': len(to_airport),
        'trips_from': len(from_airport),
        'avg_fare_to': to_airport['fare_amount'].mean(),
        'avg_fare_from': from_airport['fare_amount'].mean(),
        'avg_duration_to': to_airport['trip_duration_min'].mean(),
        'avg_duration_from': from_airport['trip_duration_min'].mean(),
        'avg_tip_to': to_airport['tip_pct'].mean(),
        'avg_tip_from': from_airport['tip_pct'].mean(),
    }

elapsed = time.time() - t0
print(f"Airport analysis computed in {elapsed:.2f}s")
print(f"\nAirport Trip Statistics:")
for airport, stats in airport_results.items():
    print(f"\n  {airport}:")
    print(f"    To airport:   {stats['trips_to']:>7,} trips, avg ${stats['avg_fare_to']:.2f} fare, {stats['avg_duration_to']:.0f}min, {stats['avg_tip_to']:.1f}% tip")
    print(f"    From airport: {stats['trips_from']:>7,} trips, avg ${stats['avg_fare_from']:.2f} fare, {stats['avg_duration_from']:.0f}min, {stats['avg_tip_from']:.1f}% tip")

In [ ]:
# Surge pricing analysis: identify hours/days with highest fares relative to average
t0 = time.time()

# Hourly-daily fare matrix
surge_matrix = df.pivot_table(
    values='fare_amount', 
    index='pickup_hour', 
    columns='pickup_day',
    aggfunc='mean'
).reindex(columns=day_order).round(2)

# Overall average fare
overall_avg = df['fare_amount'].mean()
surge_pct = ((surge_matrix - overall_avg) / overall_avg * 100).round(1)

# Find peak surge periods
max_surge_hour = surge_pct.max(axis=1).idxmax()
max_surge_day = surge_pct.max(axis=0).idxmax()
max_surge_val = surge_pct.max().max()

elapsed = time.time() - t0
print(f"Surge analysis computed in {elapsed:.2f}s")
print(f"\nSurge Pricing Matrix (% above/below average ${overall_avg:.2f}):")
print(surge_pct.to_string())
print(f"\nPeak surge: {max_surge_val:.1f}% above average at hour {max_surge_hour}")
print(f"Highest average day: {max_surge_day}")